# 03 — ESM-2 whole-protein embeddings

**Study:** Evaluating Protein Language Models for Function-Based Screening of Heme-Binding Proteins Against Traditional Sequence-Search Methods

**Question for this notebook:** Can ESM-2's representations rank heme-binding proteins above controls when compared with our 30 reference heme binders?

- Model: `facebook/esm2_t6_8M_UR50D` (six layers; 8M parameters; 320-dimensional embeddings).
- Compare each of the same **50 test proteins** with the **30 heme-binding references**.
- Original representation: **Layer 6 mean pooling**; additional representations from our pilot: **Layer 6 CLS**, **Layer 3 mean**, **Layer 3 max**.
- Pooling is across amino-acid tokens only, except CLS, which uses the first special token.
- Similarity score: **largest cosine similarity to any reference**.
- Sequences above **1,022 amino acids** are shortened as in our initial run (the affected reference is NOS3/P29474).

Run cells in order in Colab. The reference proteins are never used to train ESM-2.

In [ ]:
# Import the numerical and machine-learning packages.
import sys
import subprocess
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from IPython.display import display

# Install Transformers if the current Google Colab runtime does not have it.
try:
    import transformers
    from transformers import AutoTokenizer, AutoModel
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "transformers>=4.40,<5"
    ], check=True)
    import transformers
    from transformers import AutoTokenizer, AutoModel

# Show software versions to make our run easier to reproduce.
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

In [ ]:
# Find our 80-protein dataset beside a cloned GitHub repository, or upload it in Colab.
from pathlib import Path
import pandas as pd
from IPython.display import display

# First check common locations for the dataset.
candidates = [
    Path("../data/master_dataset_80_proteins.csv"),
    Path("data/master_dataset_80_proteins.csv"),
    Path("master_dataset_80_proteins.csv"),
]

# Use the first existing file, if available.
existing = next((path for path in candidates if path.exists()), None)

# Otherwise request a file upload in Google Colab.
if existing is None:
    from google.colab import files
    print("Upload master_dataset_80_proteins.csv")
    uploaded = files.upload()
    if "master_dataset_80_proteins.csv" not in uploaded:
        raise ValueError("Please select master_dataset_80_proteins.csv")
    existing = Path("master_dataset_80_proteins.csv")

# Read the data and make sure the records match our study design.
df = pd.read_csv(existing).reset_index(drop=True)
assert len(df) == 80, "Expected 80 protein records."
assert df["record_id"].is_unique, "Protein record IDs must be unique."
assert df["amino_acid_sequence"].notna().all(), "Missing amino-acid sequences."
assert df["dataset_group"].value_counts().to_dict() == {
    "REFERENCE": 30,
    "AI_DESIGNED_POSITIVE": 20,
    "NATURAL_POSITIVE": 10,
    "NEGATIVE_CONTROL": 20,
}, "Unexpected group sizes."
print("Loaded", len(df), "proteins from", existing)
print(df["dataset_group"].value_counts().to_string())

In [ ]:
# Load the pretrained ESM-2 model used in our pilot.
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Use a GPU when available; CPU is also supported.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print("ESM-2 model:", model_name)
print("Device:", device)

# The model has six transformer layers and 320-dimensional hidden states.
assert model.config.hidden_size == 320, "Unexpected ESM-2 model architecture."
assert model.config.num_hidden_layers == 6, "Unexpected number of ESM-2 layers."

In [ ]:
# Each test uses the same sequence input as the original ESM-2 experiments.
# The first 1,022 amino acids are kept if a protein is longer.
method_names = [
    "layer3_mean", "layer3_max", "layer6_mean", "layer6_cls",
]

# Make an empty list of protein embeddings for each representation.
collected = {name: [] for name in method_names}
truncated = []
lengths_used = []

# Process the 80 sequences once, collecting embeddings from both layers.
for i, protein in df.iterrows():
    # Get the full amino-acid sequence from our master dataset.
    full_sequence = str(protein["amino_acid_sequence"]).strip().upper()

    # Apply the 1,022-residue limit used in our earlier ESM-2 runs.
    sequence = full_sequence[:1022]
    lengths_used.append(len(sequence))
    if len(full_sequence) > 1022:
        truncated.append(str(protein["record_id"]))

    # Convert the sequence to ESM model tokens, including special tokens.
    inputs = tokenizer(sequence, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Run the pretrained model; do not update any weights.
    with torch.no_grad():
        output = model(**inputs, output_hidden_states=True)

    # Extract the amino-acid positions only (exclude BOS/EOS tokens).
    layer3 = output.hidden_states[3][0, 1:len(sequence)+1, :]
    layer6 = output.hidden_states[6][0, 1:len(sequence)+1, :]

    # Mean pooling: average the amino-acid embeddings.
    collected["layer3_mean"].append(layer3.mean(dim=0).cpu().numpy())
    collected["layer6_mean"].append(layer6.mean(dim=0).cpu().numpy())

    # Max pooling: keep the maximum value for each embedding dimension.
    collected["layer3_max"].append(layer3.max(dim=0).values.cpu().numpy())

    # CLS representation: use the first special-token embedding from Layer 6.
    collected["layer6_cls"].append(output.hidden_states[6][0, 0, :].cpu().numpy())

    # Show progress every 10 proteins.
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/80 proteins")

# Combine embeddings into numeric matrices (80 proteins x 320 dimensions).
embeddings = {name: np.stack(values) for name, values in collected.items()}
print("ESM-2 embeddings:", {name: v.shape for name, v in embeddings.items()})
print("Truncated proteins:", truncated)

In [ ]:
# Identify the 30 references and the 50 test proteins.
ref_mask = (df["dataset_group"] == "REFERENCE").to_numpy()
test_mask = ~ref_mask
references = df[ref_mask].reset_index(drop=True)
tests = df[test_mask].reset_index(drop=True)

# Keep test identifiers and labels in every result table.
scores = tests[[
    "record_id", "protein_name", "dataset_group", "heme_binding_label",
    "sequence_length",
]].copy()
scores["length_used"] = np.array(lengths_used)[test_mask]

# Compute the similarity of each test protein to all 30 references.
all_similarity_matrices = {}
for name, matrix in embeddings.items():
    # Cosine similarity compares the directions of the two embeddings.
    similarity = cosine_similarity(matrix[test_mask], matrix[ref_mask])
    all_similarity_matrices[name] = similarity

    # Each protein receives its highest reference similarity as its score.
    scores[f"{name}_similarity"] = similarity.max(axis=1)

    # Also record which reference produced the maximum score.
    best_index = similarity.argmax(axis=1)
    scores[f"{name}_reference"] = references.iloc[best_index]["record_id"].to_numpy()

# Save every score needed to compare BLASTp and the four ESM-2 variants.
scores.to_csv("esm2_embedding_method_scores.csv", index=False)

# Save all 1,500 test-to-reference similarities for original Layer-6 mean pooling.
all_mean6 = pd.DataFrame(
    all_similarity_matrices["layer6_mean"],
    index=tests["record_id"],
    columns=references["record_id"],
)
all_mean6.index.name = "test_protein_id"
all_mean6.to_csv("esm2_all_similarities.csv")

# Recreate the original final-layer mean-pooling top-match CSV.
original_top = tests[[
    "record_id", "protein_name", "dataset_group", "heme_binding_label"
]].copy()
original_top["esm2_top_reference_id"] = scores["layer6_mean_reference"]
reference_names = references.set_index("record_id")["protein_name"]
original_top["esm2_top_reference_name"] = (
    original_top["esm2_top_reference_id"].map(reference_names)
)
original_top["esm2_cosine_similarity"] = scores["layer6_mean_similarity"].round(4)
original_top.to_csv("esm2_top_matches.csv", index=False)

# Save smaller tables corresponding to our earlier Colab comparisons.
keep = ["record_id", "protein_name", "dataset_group", "heme_binding_label"]
for filename, choices in {
    "esm2_mean_vs_cls.csv": ["layer6_mean", "layer6_cls"],
    "esm2_layer3_vs_layer6.csv": ["layer3_mean", "layer6_mean"],
    "esm2_layer3_mean_vs_max.csv": ["layer3_mean", "layer3_max"],
}.items():
    # Each compact file includes both representations and their top references.
    columns = keep + [f"{name}_{field}" for name in choices for field in ["similarity", "reference"]]
    scores[columns].to_csv(filename, index=False)

# Display the main comparisons.
display(scores[["protein_name", "dataset_group", "layer6_mean_similarity", "layer3_max_similarity"]])

In [ ]:
# Calculate ROC AUC for all four ESM-2 representations and study subgroups.
summary_rows = []
for group in ["ALL", "AI_DESIGNED_POSITIVE", "NATURAL_POSITIVE"]:
    # For subgroup analyses, compare that positive group against all controls.
    if group == "ALL":
        block = scores
    else:
        block = scores[scores["dataset_group"].isin([group, "NEGATIVE_CONTROL"])]

    # Convert our existing positive/negative labels into 1/0.
    labels = block["heme_binding_label"].str.lower().eq("positive").astype(int)

    # Use maximum similarity to any reference as the continuous ranking score.
    for name in method_names:
        auc = roc_auc_score(labels, block[f"{name}_similarity"])
        summary_rows.append({"group": group, "method": name, "roc_auc": auc})

# Print and save the original representation-comparison results.
auc_summary = pd.DataFrame(summary_rows)
print("ESM-2 AUC COMPARISON")
display(auc_summary.pivot(index="method", columns="group", values="roc_auc").round(3))
auc_summary.to_csv("esm2_auc_summary.csv", index=False)

In [ ]:
# Download the files used by the performance-evaluation notebook.
from google.colab import files

# This single table includes all four ESM-2 scores for the 50 test proteins.
files.download("esm2_embedding_method_scores.csv")

# This table contains AUCs for the ESM-2 representations.
files.download("esm2_auc_summary.csv")

# This file preserves all pairwise similarities for the original mean-pooling method.
files.download("esm2_all_similarities.csv")

# Original top matches are useful for checking against the first Colab run.
files.download("esm2_top_matches.csv")

### Outputs and recorded pilot results

`esm2_embedding_method_scores.csv` contains the four methods' scores and closest references for all 50 test proteins. `esm2_auc_summary.csv` records method-by-group AUC. The notebook also generates the initial mean-pooling top-match and similarity-matrix files.

**Observed overall AUCs:** Layer-6 mean 0.517; Layer-6 CLS 0.542; Layer-3 mean 0.542; Layer-3 max 0.753.